# Gurobi Model 2 — Selective Multi-product VRPTW based on Devin's model

I keep the same parameter in Devin's model, and added some new parameter to creat time window for cafes'.
1. Load data
2. Sets
3. Parameters(New)
4. Build model
5. Objective function
6. Constraints
7. Solve
8. Print solution

The main differences from Model 1 are: Model 2 adds customer selection, vehicle departure time, arrival time, cafe pickup time window, time propagation constraint, soft time window late penalty, inventory constraint, and vehicle-product compatibility constraint.

In [43]:
import pandas as pd
import numpy as np
from pathlib import Path

import gurobipy as gp
from gurobipy import GRB


# 1. Load data

In [44]:
DATA_REAL = "../data/real/"
DATA_SYN = "../data/synthetic/"

distance_df = pd.read_csv(DATA_REAL + "distance_matrix.csv", index_col=0)
time_df = pd.read_csv(DATA_REAL + "time_matrix.csv", index_col=0)
cafes_df = pd.read_csv(DATA_SYN + "cafes.csv")
demands_df = pd.read_csv(DATA_SYN + "demands.csv")
products_df = pd.read_csv(DATA_SYN + "products.csv")
vans_df = pd.read_csv(DATA_SYN + "vans.csv")
depot_df = pd.read_csv(DATA_SYN + "depot.csv")
perish_df = pd.read_csv(DATA_SYN + "perishability_params.csv")

print("distance_df:", distance_df.shape)
print("time_df:", time_df.shape)
print("cafes_df:", cafes_df.shape)
print("demands_df:", demands_df.shape)
print("products_df:", products_df.shape)
print("vans_df:", vans_df.shape)


distance_df: (35, 35)
time_df: (35, 35)
cafes_df: (34, 5)
demands_df: (238, 3)
products_df: (7, 10)
vans_df: (5, 4)


# 2. Experiment settings

In [45]:
# Test parameter
N_CAFES = 34             # maxminum cafes' 14,24,34

N_VANS = 5               # maxminum vans

SUBSET_STRATEGY = "closest"   # closest / first

# Model 2's extra parameter
TIME_WINDOW_MODE = "wide"     # wide / size_based / staggered (you can try size_based or staggered by yourself)
# TIME_WINDOW_MODE = "size_based"
# TIME_WINDOW_MODE = "staggered"


FORCE_ALL_CAFES = False       # True: Forced service to all cafes, making it easier to compare with Model 1.

SOFT_TIME_WINDOWS = True      # True: Lateness is allowed, but a penalty will be imposed; False: Lateness is not allowed.

ALLOW_PARTIAL_DELIVERY = False # see in cell 8

MIN_SERVICE_FRACTION = 1.0    # If ALLOW_PARTIAL_DELIVERY=True, what is the minimum percentage to send after selecting the cafe?

MIN_CAFES_SERVED = 1          # To avoid not serving any cafes due to excessively high fixed costs.

# Cost and solution parameters
FIXED_VAN_COST = 30.0         # Fixed costs of starting up a van

BETA_ROUTE = 0.0              # # Long route penalty. To penalize long routes, we can set from 1-5 etc.

LATENESS_PENALTY = 25.0       # Late hour penalty

TIME_LIMIT = 300              # Gurobi Time Limit

MIP_GAP = 0.01                # Gap limit

# If not None, override T_max_hours in perishability_params.csv
T_MAX_OVERRIDE = None         # 2.0, 3.0, 4.0


# 3. Sets

In [46]:
# Sets

# Check path
DATA_SYN = Path(DATA_SYN)

# read data
cafes_df = pd.read_csv(DATA_SYN / "cafes.csv")
vans_df = pd.read_csv(DATA_SYN / "vans.csv")
products_df = pd.read_csv(DATA_SYN / "products.csv")
depot_df = pd.read_csv(DATA_SYN / "depot.csv")

# depot 
depot_name = depot_df.loc[0, "name"]

# cafes
cafes = cafes_df["cafe_name"].tolist()

# nodes：depot + cafes
nodes = [depot_name] + cafes

#  vans
K = vans_df["van_id"].tolist()

# products
P = products_df["product_id"].tolist()

# milk products
milk_products = products_df.loc[
    products_df["category"] == "milk",
    "product_id"
].tolist()

# perishable products
perishable_flag = products_df["perishable"].astype(str).str.lower().isin(
    ["true", "1", "yes"]
)

perishable_products = products_df.loc[
    perishable_flag,
    "product_id"
].tolist()

# directed arcs
A = [(i, j) for i in nodes for j in nodes if i != j]

print("Depot:", depot_name)

print("Number of cafes:", len(cafes))

print("Number of vans:", len(K))

print("Number of products:", len(P))

print("Milk products:", milk_products)

print("Perishable products:", perishable_products)

print("Number of arcs:", len(A))






Depot: Peter Hall Building UniMelb
Number of cafes: 34
Number of vans: 5
Number of products: 7
Milk products: ['milk_full', 'milk_oat', 'milk_soy', 'milk_almond']
Perishable products: ['milk_full', 'milk_oat', 'milk_soy', 'milk_almond']
Number of arcs: 1190


# 4. Parameters

In [47]:

# Check if parameter matches model 1


missing_distance = (set(nodes) - set(distance_df.index)) | (set(nodes) - set(distance_df.columns))

missing_time = (set(nodes) - set(time_df.index)) | (set(nodes) - set(time_df.columns))

if missing_distance:
    raise ValueError(f"distance_matrix.csv These nodes are missing: {missing_distance}")
    
if missing_time:
    raise ValueError(f"time_matrix.csv These nodes are missing: {missing_time}")

if cafes_df["cafe_name"].duplicated().any():
    duplicated = cafes_df.loc[cafes_df["cafe_name"].duplicated(), "cafe_name"].tolist()
    raise ValueError(f"Duplicate cafe_names exist, so cafe name cannot be used directly for modeling: {duplicated}")

print("Data consistency checks passed.")


Data consistency checks passed.


In [48]:

# 4.1 Demand parameters d[i,p]

# demands.csv use cafe_id，however, the route nodes use cafe_name，mapping first。
demand_raw = {
    (row["cafe_id"], row["product_id"]): float(row["daily_demand"])
    for _, row in demands_df.iterrows()
}

cafe_id_to_name = dict(zip(cafes_df["cafe_id"], cafes_df["cafe_name"]))

# d[i,p]：The requirement of cafe i for product p. The default value is 0 if no record is found.
d = {}
for cafe_id, cafe_name in cafe_id_to_name.items():
    for p in P:
        d[cafe_name, p] = demand_raw.get((cafe_id, p), 0.0)


# 4.2 Product parameters

w = dict(zip(products_df["product_id"], products_df["weight_per_unit_kg"]))
revenue = dict(zip(products_df["product_id"], products_df["revenue_per_unit"]))
cost = dict(zip(products_df["product_id"], products_df["cost_per_unit"]))
margin = dict(zip(products_df["product_id"], products_df["margin_per_unit"]))
perishable = dict(zip(products_df["product_id"], products_df["perishable"].astype(bool)))
shelf_life = dict(zip(products_df["product_id"], products_df["shelf_life_hours"]))


# 4.3 Van parameters

Q = dict(zip(vans_df["van_id"], vans_df["capacity_kg"]))
F = dict(zip(vans_df["van_id"], vans_df["fuel_cost_per_km"]))
has_refrigeration = dict(zip(vans_df["van_id"], vans_df["has_refrigeration"].astype(bool)))


# alpha[k,p] = 1 indicates that van k can deliver product p.

# Default logic here: all vans can deliver non-perishable products;

# However, perishable products require a refrigerated van.
alpha = {}
for k in K:
    for p in P:
        if perishable[p]:
            alpha[k, p] = 1 if has_refrigeration[k] else 0
        else:
            alpha[k, p] = 1


# 4.4 Distance and time parameters
D = {}
T = {}

for i, j in A:
    D[i, j] = float(distance_df.loc[i, j])
    # time_matrix.csv convert min to hour
    T[i, j] = float(time_df.loc[i, j]) / 60.0
    


# 4.5 Perishability and service parameters
perish_params = dict(zip(perish_df["parameter"], perish_df["value"]))
T_max = float(perish_params.get("T_max_hours", 3.0))

if T_MAX_OVERRIDE is not None:
    T_max = float(T_MAX_OVERRIDE)

loading_time = float(perish_params.get("loading_time_minutes", 15.0)) / 60.0

service_time = float(perish_params.get("service_time_minutes", 5.0)) / 60.0

# Inventory I[p]. Currently, there is no separate inventory file.
# The default inventory is exactly equal to the total demand for all selected cafes.
# Replace real inventory data later
I = {p: sum(d[i, p] for i in cafes) for p in P}

# Big-M time constant. The unit is hours, not |C|.
DAY_START = 7.0
DAY_END = 17.0
M_time = 24.0

# Total milk volume (Big-M), used to activate g[k]
M_milk = sum(d[i, p] for i in cafes for p in milk_products)

print("T_max:", T_max)
print("loading_time:", loading_time)
print("service_time:", service_time)
print("Total selected demand weight:", sum(w[p] * d[i, p] for i in cafes for p in P))


T_max: 3.0
loading_time: 0.25
service_time: 0.08333333333333333
Total selected demand weight: 1207.73


# 5. Generate time windows

In [49]:
# # Model 2 is a VRPTW model (Receiving Time Windows), 
# So generate time window [a_i, b_i] for each cafe.
# wide for debug；size_based and staggered for analysis 

def generate_time_windows(cafes_df, mode="wide"):
    tw = cafes_df[["cafe_id", "cafe_name", "size"]].copy()

    if mode == "wide":
        tw["tw_start"] = 7.0
        tw["tw_end"] = 17.0

    elif mode == "size_based":
        # Larger cafes generally prefer to receive milk as soon as possible, while smaller cafes offer more flexible delivery times.
        window_map = {
            "large": (7.0, 10.5),
            "medium": (8.0, 12.5),
            "small": (9.0, 14.0),
        }
        tw["tw_start"] = tw["size"].map(lambda s: window_map.get(s, (8.0, 14.0))[0])
        tw["tw_end"] = tw["size"].map(lambda s: window_map.get(s, (8.0, 14.0))[1])

    elif mode == "staggered":
        # Batch time windows for more complex VRPTW tests.
        starts = []
        ends = []
        for idx, _ in tw.reset_index(drop=True).iterrows():
            block = idx % 3
            if block == 0:
                starts.append(7.0)
                ends.append(10.5)
            elif block == 1:
                starts.append(9.0)
                ends.append(12.5)
            else:
                starts.append(11.0)
                ends.append(15.0)
        tw["tw_start"] = starts
        tw["tw_end"] = ends

    else:
        raise ValueError("TIME_WINDOW_MODE must be 'wide', 'size_based', or 'staggered'.")

    return tw





time_windows_df = generate_time_windows(cafes_df, mode=TIME_WINDOW_MODE)

a = dict(zip(time_windows_df["cafe_name"], time_windows_df["tw_start"]))
b = dict(zip(time_windows_df["cafe_name"], time_windows_df["tw_end"]))

# Save data

DATA_SYN = Path(DATA_SYN)
DATA_SYN.mkdir(parents=True, exist_ok=True)
time_window_output = DATA_SYN / f"model3_time_windows_{TIME_WINDOW_MODE}.csv"
time_windows_df.to_csv(time_window_output, index=False)
print("Time windows saved to:", time_window_output)

display(time_windows_df.head())


Time windows saved to: ..\data\synthetic\model3_time_windows_wide.csv


,cafe_id,cafe_name,size,tw_start,tw_end
0,cafe_01,Proud Mary Coffee,small,7.0,17.0
1,cafe_02,Patricia Coffee Brewers,large,7.0,17.0
2,cafe_03,Seven Seeds Coffee Roasters,medium,7.0,17.0
3,cafe_04,Industry Beans Fitzroy,medium,7.0,17.0
4,cafe_05,Market Lane Coffee HQ,small,7.0,17.0


# 6. Build model

In [50]:
m = gp.Model("model3_selective_multiproduct_vrptw")

# Routing variables
x = m.addVars(A, K, vtype=GRB.BINARY, name="x")

# Service variables
y = m.addVars(cafes, K, vtype=GRB.BINARY, name="y")
z = m.addVars(cafes, vtype=GRB.BINARY, name="z")

# Delivery quantity variables
q = m.addVars(cafes, P, K, lb=0, vtype=GRB.CONTINUOUS, name="q")

# Route duration and milk indicator variables
R = m.addVars(K, lb=0, vtype=GRB.CONTINUOUS, name="R")
g = m.addVars(K, vtype=GRB.BINARY, name="g")

# New added: departure time, arrival time, lateness
# departure[k]: The time when van k departs from depot
# tau[i,k]: The arrival time when van k reaches cafe i
# late[i,k]: if we use soft time windows, this means delay hours
departure = m.addVars(K, lb=0, vtype=GRB.CONTINUOUS, name="departure")
tau = m.addVars(cafes, K, lb=0, vtype=GRB.CONTINUOUS, name="tau")
late = m.addVars(cafes, K, lb=0, vtype=GRB.CONTINUOUS, name="late")


# 7. Objective function

In [51]:
# Product Profit: Model 2 uses margin instead of simple revenue.
product_margin = gp.quicksum(
    margin[p] * q[i, p, k]
    for i in cafes
    for p in P
    for k in K
)

# Transportation Costs: Following Devin's format, fuel cost per km * distance * x
transport_cost = gp.quicksum(
    F[k] * D[i, j] * x[i, j, k]
    for (i, j) in A
    for k in K
)

# Enable fixed vehicle cost: If the van k originates from the depot, a fixed cost will be charged.
fixed_cost = gp.quicksum(
    FIXED_VAN_COST * gp.quicksum(x[depot_name, j, k] for j in cafes)
    for k in K
)

# Long route penalty: Can be set to 0, or used to penalize longer routes.(beta = 5.0 in Devin's model)
route_penalty = gp.quicksum(
    BETA_ROUTE * R[k]
    for k in K
)

# Soft time window lateness penalty
lateness_cost = gp.quicksum(
    LATENESS_PENALTY * late[i, k]
    for i in cafes
    for k in K
)

m.setObjective(
    product_margin - transport_cost - fixed_cost - route_penalty - lateness_cost,
    GRB.MAXIMIZE
)


# 8. Constraints

In [52]:
# 8.1 Each cafe served by at most one van
m.addConstrs(
    (gp.quicksum(y[i, k] for k in K) == z[i]
     for i in cafes),
    name="serve_once"
)

# 8.2 Demand satisfaction
# see cell 2
if ALLOW_PARTIAL_DELIVERY:
    # allow partial deliver: After choose a cafe, the delivery volume can be lower than the total demand, but it must meet the minimum ratio.
    m.addConstrs(
        (gp.quicksum(q[i, p, k] for k in K) <= d[i, p] * z[i]
         for i in cafes
         for p in P),
        name="demand_upper"
    )
    m.addConstrs(
        (gp.quicksum(q[i, p, k] for k in K) >= MIN_SERVICE_FRACTION * d[i, p] * z[i]
         for i in cafes
         for p in P),
        name="demand_lower"
    )
else:
    # Default: Selecting cafe requires meeting all requirements.
    m.addConstrs(
        (gp.quicksum(q[i, p, k] for k in K) == d[i, p] * z[i]
         for i in cafes
         for p in P),
        name="demand_satisfaction"
    )

# 8.3 Deliveries only if van services cafe + vehicle-product compatibility
# when alpha[k,p]=0, van k cannot deliver product p。
m.addConstrs(
    (q[i, p, k] <= d[i, p] * alpha[k, p] * y[i, k]
     for i in cafes
     for p in P
     for k in K),
    name="delivery_if_served_and_compatible"
)

# 8.4 Vehicle capacity
m.addConstrs(
    (gp.quicksum(w[p] * q[i, p, k]
                 for i in cafes
                 for p in P) <= Q[k]
     for k in K),
    name="capacity"
)

# 8.5 Product inventory
m.addConstrs(
    (gp.quicksum(q[i, p, k]
                 for i in cafes
                 for k in K) <= I[p]
     for p in P),
    name="inventory"
)

# 8.6 Flow conservation
m.addConstrs(
    (gp.quicksum(x[i, j, k] for j in nodes if j != i) == y[i, k]
     for i in cafes
     for k in K),
    name="flow_out"
)

m.addConstrs(
    (gp.quicksum(x[j, i, k] for j in nodes if j != i) == y[i, k]
     for i in cafes
     for k in K),
    name="flow_in"
)

# 8.7 Depot departure and return
m.addConstrs(
    (gp.quicksum(x[depot_name, j, k] for j in cafes) <= 1
     for k in K),
    name="depot_departure"
)

m.addConstrs(
    (gp.quicksum(x[i, depot_name, k] for i in cafes) <= 1
     for k in K),
    name="depot_return"
)

m.addConstrs(
    (gp.quicksum(x[depot_name, j, k] for j in cafes)
     ==
     gp.quicksum(x[i, depot_name, k] for i in cafes)
     for k in K),
    name="depot_balance"
)


{'van_1': <gurobi.Constr *Awaiting Model Update*>,
 'van_2': <gurobi.Constr *Awaiting Model Update*>,
 'van_3': <gurobi.Constr *Awaiting Model Update*>,
 'van_4': <gurobi.Constr *Awaiting Model Update*>,
 'van_5': <gurobi.Constr *Awaiting Model Update*>}

In [53]:
# 8.8 Route duration definition
# R[k] = depot loading time + travel time + service time
m.addConstrs(
    (R[k] ==
     loading_time * gp.quicksum(x[depot_name, j, k] for j in cafes)
     +
     gp.quicksum(T[i, j] * x[i, j, k] for (i, j) in A)
     +
     service_time * gp.quicksum(y[i, k] for i in cafes)
     for k in K),
    name="route_duration"
)

# 8.9 Milk indicator
m.addConstrs(
    (gp.quicksum(q[i, p, k]
                 for i in cafes
                 for p in milk_products)
     <= M_milk * g[k]
     for k in K),
    name="milk_indicator"
)

# Not show in Latex 
# If van has not started, then g[k] does not need to be 1. (clean output)
m.addConstrs(
    (g[k] <= gp.quicksum(x[depot_name, j, k] for j in cafes)
     for k in K),
    name="milk_indicator_clean"
)

# 8.10 Maximum route duration if carrying milk
m.addConstrs(
    (R[k] <= T_max + M_time * (1 - g[k])
     for k in K),
    name="milk_time_limit"
)

{'van_1': <gurobi.Constr *Awaiting Model Update*>,
 'van_2': <gurobi.Constr *Awaiting Model Update*>,
 'van_3': <gurobi.Constr *Awaiting Model Update*>,
 'van_4': <gurobi.Constr *Awaiting Model Update*>,
 'van_5': <gurobi.Constr *Awaiting Model Update*>}

In [54]:
# 8.11 Departure time bounds
# If van has not departed, departure[k] will be pushed down to 0; 
# If it has departed, it will be between DAY_START and DAY_END.
m.addConstrs(
    (departure[k] >= DAY_START * gp.quicksum(x[depot_name, j, k] for j in cafes)
     for k in K),
    name="departure_lower"
)

m.addConstrs(
    (departure[k] <= DAY_END * gp.quicksum(x[depot_name, j, k] for j in cafes)
     for k in K),
    name="departure_upper"
)

# 12. Cafe time windows
m.addConstrs(
    (tau[i, k] >= a[i] * y[i, k]
     for i in cafes
     for k in K),
    name="time_window_start"
)

m.addConstrs(
    (tau[i, k] <= b[i] + late[i, k] + M_time * (1 - y[i, k])
     for i in cafes
     for k in K),
    name="time_window_end"
)

# Not shown in latex
# If cafe is not served by van k, then tau[i,k] is pushed down to 0
m.addConstrs(
    (tau[i, k] <= M_time * y[i, k]
     for i in cafes
     for k in K),
    name="arrival_clean"
)

if not SOFT_TIME_WINDOWS:
    # hard time windows constraints check 
    m.addConstrs(
        (late[i, k] == 0
         for i in cafes
         for k in K),
        name="no_lateness"
    )

# 13. Time propagation from depot to first cafe
m.addConstrs(
    (tau[j, k] >= departure[k] + loading_time + T[depot_name, j]
     - M_time * (1 - x[depot_name, j, k])
     for j in cafes
     for k in K),
    name="time_from_depot"
)

# 14. Time propagation between cafes
m.addConstrs(
    (tau[j, k] >= tau[i, k] + service_time + T[i, j]
     - M_time * (1 - x[i, j, k])
     for i in cafes
     for j in cafes
     if i != j
     for k in K),
    name="time_between_cafes"
)

# 15. Return to depot before route duration and workday end
m.addConstrs(
    (tau[i, k] + service_time + T[i, depot_name]
     <= departure[k] + R[k] + M_time * (1 - x[i, depot_name, k])
     for i in cafes
     for k in K),
    name="return_time_consistency"
)

m.addConstrs(
    (departure[k] + R[k]
     <= DAY_END + M_time * (1 - gp.quicksum(x[depot_name, j, k] for j in cafes))
     for k in K),
    name="return_before_day_end"
)


{'van_1': <gurobi.Constr *Awaiting Model Update*>,
 'van_2': <gurobi.Constr *Awaiting Model Update*>,
 'van_3': <gurobi.Constr *Awaiting Model Update*>,
 'van_4': <gurobi.Constr *Awaiting Model Update*>,
 'van_5': <gurobi.Constr *Awaiting Model Update*>}

In [55]:
# 16. Customer selection controls
if FORCE_ALL_CAFES:
    # Compare with Devin's model
    m.addConstrs(
        (z[i] == 1 for i in cafes),
        name="force_all_cafes"
    )
else:
    # To prevent a cafe from not serving any customers due to excessively high costs.
    m.addConstr(
        gp.quicksum(z[i] for i in cafes) >= MIN_CAFES_SERVED,
        name="minimum_selected_cafes"
    )

# Note: Here, the model does not use MTZ as the subcycle elimination method.

# Because the time propagation constraint implicitly excludes cafe-only subtours:

# For example:

# If A->B->C->A are selected simultaneously, it would imply tau_A >= tau_A + positive travel(service time), a contradiction.

# Therefore, the u[i,k] variable and MTZ constraint are not added here.


# 9. Solve

In [56]:
m.Params.OutputFlag = 1
m.Params.TimeLimit = TIME_LIMIT
m.Params.MIPGap = MIP_GAP

m.optimize()


Set parameter OutputFlag to value 1
Set parameter TimeLimit to value 300
Set parameter MIPGap to value 0.01
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: 13th Gen Intel(R) Core(TM) i5-13500H, instruction set [SSE2|AVX|AVX2]
Thread count: 12 physical cores, 16 logical processors, using up to 16 threads

Non-default parameters:
TimeLimit  300
MIPGap  0.01

Optimize a model with 8325 rows, 7699 columns and 45609 nonzeros (Max)
Model fingerprint: 0x1624c18e
Model has 7310 linear objective coefficients
Variable types: 1540 continuous, 6159 integer (6159 binary)
Coefficient statistics:
  Matrix range     [7e-03, 1e+03]
  Objective range  [7e-02, 3e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 7e+02]

Presolve removed 1440 rows and 1185 columns
Presolve time: 1.43s
Presolved: 6885 rows, 6514 columns, 58202 nonzeros
Variable types: 350 continuous, 6164 integer (6164 binary)
Found heuristic solution: objective 1617.8080500

Ro

# 10. Print solution

In [57]:
def status_name(model):
    status_map = {
        GRB.OPTIMAL: "OPTIMAL",
        GRB.TIME_LIMIT: "TIME_LIMIT",
        GRB.INFEASIBLE: "INFEASIBLE",
        GRB.UNBOUNDED: "UNBOUNDED",
        GRB.INF_OR_UNBD: "INF_OR_UNBD",
    }
    return status_map.get(model.status, str(model.status))


def hour_to_clock(t):
    if t is None or pd.isna(t):
        return ""
    h = int(t)
    mnt = int(round((t - h) * 60))
    if mnt == 60:
        h += 1
        mnt = 0
    return f"{h:02d}:{mnt:02d}"


def extract_route(k):
    # Starting from depot, extract the main route of van k based on the variable x.
    route = [depot_name]
    current = depot_name
    visited = {depot_name}

    while True:
        next_nodes = [
            j for j in nodes
            if j != current and x[current, j, k].X > 0.5
        ]

        if not next_nodes:
            break

        nxt = next_nodes[0]
        route.append(nxt)

        if nxt == depot_name:
            break

        if nxt in visited:
            # In theory, time propagation should exclude this situation.
            route.append("SUBTOUR_WARNING")
            break

        visited.add(nxt)
        current = nxt

    return route


if m.SolCount > 0:
    print("Status:", status_name(m))
    print("Objective:", m.ObjVal)
    print("MIP gap:", getattr(m, "MIPGap", None))
    print("Runtime:", getattr(m, "Runtime", None))

    selected_cafes = [i for i in cafes if z[i].X > 0.5]
    print("Selected cafes:", len(selected_cafes))

    total_margin = product_margin.getValue()
    total_transport_cost = transport_cost.getValue()
    total_fixed_cost = fixed_cost.getValue()
    total_lateness_cost = lateness_cost.getValue()
    total_route_penalty = route_penalty.getValue()

    print("Total margin:", round(total_margin, 4))
    print("Transport cost:", round(total_transport_cost, 4))
    print("Fixed van cost:", round(total_fixed_cost, 4))
    print("Lateness cost:", round(total_lateness_cost, 4))
    print("Route penalty:", round(total_route_penalty, 4))

    route_rows = []
    delivery_rows = []

    for k in K:
        used = sum(x[depot_name, j, k].X for j in cafes) > 0.5
        route = extract_route(k) if used else [depot_name]

        total_weight = sum(
            w[p] * q[i, p, k].X
            for i in cafes
            for p in P
        )

        route_rows.append({
            "van": k,
            "used": int(used),
            "departure_time": departure[k].X if used else 0.0,
            "departure_clock": hour_to_clock(departure[k].X) if used else "",
            "route_duration_hr": R[k].X if used else 0.0,
            "carries_milk": int(round(g[k].X)),
            "load_kg": total_weight,
            "capacity_kg": Q[k],
            "route": " -> ".join(route),
        })

        for i in cafes:
            if y[i, k].X > 0.5:
                for p in P:
                    val = q[i, p, k].X
                    if val > 1e-6:
                        delivery_rows.append({
                            "cafe": i,
                            "van": k,
                            "product": p,
                            "quantity": val,
                            "weight_kg": w[p] * val,
                            "arrival_time": tau[i, k].X,
                            "arrival_clock": hour_to_clock(tau[i, k].X),
                            "tw_start": a[i],
                            "tw_end": b[i],
                            "late_hr": late[i, k].X,
                        })

    route_table = pd.DataFrame(route_rows)
    delivery_table = pd.DataFrame(delivery_rows)

    pd.set_option("display.max_colwidth", 120)
    display(route_table)
    display(delivery_table.head(30))

else:
    print("Model was not solved successfully.")
    print("Status:", status_name(m))


Status: OPTIMAL
Objective: 2832.68905
MIP gap: 0.009365068924113587
Runtime: 3.5199999809265137
Selected cafes: 34
Total margin: 2926.5
Transport cost: 33.8109
Fixed van cost: 60.0
Lateness cost: 0.0
Route penalty: 0.0


,van,used,departure_time,departure_clock,route_duration_hr,carries_milk,load_kg,capacity_kg,route
0,van_1,0,0.0,,0.000000,0,0.00,600,Peter Hall Building UniMelb
1,van_2,0,0.0,,0.000000,0,0.00,600,Peter Hall Building UniMelb
2,van_3,1,7.0,07:00,2.788637,1,619.25,700,Peter Hall Building UniMelb -> MAKER Coffee South Yarra -> Market Lane Prahran Market -> Square Lane Coffee -> Vacat...
3,van_4,1,7.0,07:00,2.696135,1,588.48,700,Peter Hall Building UniMelb -> Core Roasters -> Market Lane Coffee HQ -> Standing Room Fitzroy North -> Industry Bea...
4,van_5,0,0.0,,0.000000,0,0.00,500,Peter Hall Building UniMelb


,cafe,van,product,quantity,weight_kg,arrival_time,arrival_clock,tw_start,tw_end,late_hr
0,Patricia Coffee Brewers,van_3,milk_full,51.0,52.53,8.438750,08:26,7.0,17.0,0.0
1,Patricia Coffee Brewers,van_3,milk_oat,12.0,12.36,8.438750,08:26,7.0,17.0,0.0
2,Patricia Coffee Brewers,van_3,milk_soy,7.0,7.14,8.438750,08:26,7.0,17.0,0.0
3,Patricia Coffee Brewers,van_3,milk_almond,6.0,6.12,8.438750,08:26,7.0,17.0,0.0
4,Patricia Coffee Brewers,van_3,bean_house,5.0,5.00,8.438750,08:26,7.0,17.0,0.0
5,Patricia Coffee Brewers,van_3,bean_single,3.0,3.00,8.438750,08:26,7.0,17.0,0.0
6,Patricia Coffee Brewers,van_3,bean_decaf,2.0,2.00,8.438750,08:26,7.0,17.0,0.0
7,ST. ALi Coffee Roasters,van_3,milk_full,16.0,16.48,8.279000,08:17,7.0,17.0,0.0
8,ST. ALi Coffee Roasters,van_3,milk_oat,3.0,3.09,8.279000,08:17,7.0,17.0,0.0
9,ST. ALi Coffee Roasters,van_3,milk_soy,3.0,3.06,8.279000,08:17,7.0,17.0,0.0


# 11. comparison with Model 

To compare with Devin's model, change these variables.

```python
FORCE_ALL_CAFES = True
TIME_WINDOW_MODE = "wide"
FIXED_VAN_COST = 0.0
BETA_ROUTE = 0.0
LATENESS_PENALTY = 0.0
```

Then rerun all cells starting from Section 3. This way, Model 3 will force all cafes to be served and minimize the impact of additional business selection factors.